In [28]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


CUDA available: True
CUDA version: 12.1
GPU count: 1
GPU 0: NVIDIA GeForce RTX 3060


In [29]:
import os
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl.trainer.sft_trainer import SFTTrainer


In [31]:
# Define the model ID
model_id = "Qwen/Qwen2-0.5B-Instruct"

# Configure BitsAndBytesConfig for QLoRa
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [32]:
# Load the base model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set the padding token. This is a common practice for models that
# do not have a dedicated pad_token.
tokenizer.pad_token = tokenizer.eos_token

In [33]:
from datasets import Dataset, DatasetDict
from typing import cast

dataset_name = "ruslanmv/ai-medical-chatbot"
dataset = load_dataset(dataset_name, split="train")

# Handle DatasetDict case
if isinstance(dataset, DatasetDict):
    dataset = dataset["train"]

# Assert this is a Dataset (not IterableDataset)
dataset = cast(Dataset, dataset)
dataset = dataset.shuffle(seed=42)

print(dataset)


Dataset({
    features: ['Description', 'Patient', 'Doctor'],
    num_rows: 256916
})


In [34]:
# Define the QLoRa configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [35]:
model_id = "Qwen/Qwen2-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation="flash_attention_2" # only works in wsl or linux
)


In [36]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [37]:
SYSTEM_PROMPT = """You are a helpful AI medical assistant. 
Please provide informative and safe medical advice. 
Do not provide a diagnosis. Advise the user to consult a professional."""

In [39]:
def format_chat_template(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["Patient"]},
        {"role": "assistant", "content": row["Doctor"]},
    ]
    return {"messages": messages}


In [40]:
processed_dataset = dataset.map(
    format_chat_template,
    remove_columns=dataset.features
)

print(processed_dataset)


Dataset({
    features: ['messages'],
    num_rows: 256916
})


In [41]:
print(processed_dataset["messages"])

Column([[{'content': 'You are a helpful AI medical assistant. \nPlease provide informative and safe medical advice. \nDo not provide a diagnosis. Advise the user to consult a professional.', 'role': 'system'}, {'content': 'last year my wife was went through a surgery for appendix cancer, that appendix was removed , that appendix slice tested in lab and found so called adino carcinoma in apedix,  after that doctor decided to operate again and remove her partial intestine, there was no sign of cancer in any test other than the biopsy of appendix, however  after one moth of hospitalization came back to home, 6 month of follow up check up no bad sign, now almost one year of surgery puss mark notice at steches near belly button .Please advice this is not a sign of any cancer', 'role': 'user'}, {'content': 'Hi and welcome to HCM. First, you dont have to worry. This cant be tumour relaps because this is lesion in abdominall wall,obviously some local infection or wound abscess.This is often se

In [42]:
training_dataset = processed_dataset.select(range(900))
eval_dataset = processed_dataset.select(range(900, 1000))
print(training_dataset)

Dataset({
    features: ['messages'],
    num_rows: 900
})


In [43]:
# Define the LoRa configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear" #This need to be checked
)

# Define the Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen2-medical-chatbot-v1",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",       # QLoRa-specific optimizer [9]
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps", #previously used "epoch"
    save_steps=50,
    save_total_limit=3,  # Keep only best 3 checkpoints
    load_best_model_at_end=True,  # Load best model
    bf16=True,                      # Use bfloat16 for training
    report_to="tensorboard",
)

In [44]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args= training_args,
    train_dataset=training_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
)



RuntimeError: TensorBoardCallback requires tensorboard to be installed. Either update your PyTorch version or install tensorboardX.

In [28]:
# Start the training process
print("Starting model training...")
trainer.train()



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting model training...


Step,Training Loss
10,2.925000
20,2.814300
30,2.759700
40,2.808200
50,2.790300
60,2.726300


TrainOutput(global_step=63, training_loss=2.8034020681229848, metrics={'train_runtime': 225.3299, 'train_samples_per_second': 4.438, 'train_steps_per_second': 0.28, 'total_flos': 763408933969920.0, 'train_loss': 2.8034020681229848, 'entropy': 2.812171792984009, 'num_tokens': 244007.0, 'mean_token_accuracy': 0.43754203915596007, 'epoch': 1.0})

In [29]:
# Save the final adapter weights
adapter_path = "./qwen2-medical-adapter"
trainer.save_model(adapter_path)
print(f"Training complete. Adapter saved to {adapter_path}")

Training complete. Adapter saved to ./qwen2-medical-adapter


In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Define paths and device
model_id = "./qwen2-medical-chatbot-v1"
adapter_path = "./qwen2-medical-adapter"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model for inference...")

# Load the 4-bit base model
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config_inf,
    device_map="auto"
)

# Load LoRa adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()  # Important for inference speed

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded. Running test inference...")

# SYSTEM + USER MESSAGE (ChatML Format)
SYSTEM_PROMPT = (
    "You are a helpful AI medical assistant. "
    "Provide safe and informative advice. "
    "Do not diagnose. Always recommend consulting a medical professional."
)

new_patient_query = "আমার পেটে ব্যাথা, আমি কি অসুধ খেতে পারি?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": new_patient_query},
]

# Apply Chat template
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # adds <|assistant|>
)

# Tokenize input
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7
)

# Decode
response_text = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\n--- Test Query ---")
print(new_patient_query)
print("\n--- Model Response ---")
print(response_text)


Loading model for inference...


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./qwen2-medical-chatbot-v1.

In [37]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model_id = "Qwen/Qwen2-0.5B-Instruct"
adapter_path = "./qwen2-medical-adapter"

# Load base model (no need for 4bit here)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)

# Load adapter into base model
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

# Push adapter to HuggingFace Hub
peft_model.push_to_hub("fahad1770/qwen2-medical-lora")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/fahad1770/qwen2-medical-lora/commit/5a558520f53ec05009912546f0415513ff99a14a', commit_message='Upload model', commit_description='', oid='5a558520f53ec05009912546f0415513ff99a14a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/fahad1770/qwen2-medical-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='fahad1770/qwen2-medical-lora'), pr_revision=None, pr_num=None)

In [ ]:
# For letter usage

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
model = PeftModel.from_pretrained(model, "fahad1770/qwen2-medical-lora")
